In [15]:
import pandas as pd
import numpy as np


from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

import warnings

In [2]:
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format) 
pd.set_option('display.max_columns', None)              
pd.set_option('display.width', None) 

In [3]:
# Load the dataset
df = pd.read_csv('train_cleaned.csv')
df_test = pd.read_csv('test_cleaned.csv')

In [4]:
df = pd.get_dummies(df, columns=['Type']) 

In [5]:

df['MarkDown1'] = df['MarkDown1'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown2'] = df['MarkDown2'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown3'] = df['MarkDown3'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown4'] = df['MarkDown4'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown5'] = df['MarkDown5'].apply(lambda x: 0 if x < 0 else x)

In [6]:
df

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size,IsSuperbowl,IsLaborDay,Ischristmas,IsThanksgiving,year,month,day,Type_A,Type_B,Type_C
0,1,1,2010-02-05,"24,924.50",0,42.31,2.57,0.00,0.00,0.00,0.00,0.00,211.10,8.11,151315,0,0,0,0,2010,2,5,True,False,False
1,1,1,2010-02-12,"46,039.49",1,38.51,2.55,0.00,0.00,0.00,0.00,0.00,211.24,8.11,151315,1,0,0,0,2010,2,12,True,False,False
2,1,1,2010-02-19,"41,595.55",0,39.93,2.51,0.00,0.00,0.00,0.00,0.00,211.29,8.11,151315,0,0,0,0,2010,2,19,True,False,False
3,1,1,2010-02-26,"19,403.54",0,46.63,2.56,0.00,0.00,0.00,0.00,0.00,211.32,8.11,151315,0,0,0,0,2010,2,26,True,False,False
4,1,1,2010-03-05,"21,827.90",0,46.50,2.62,0.00,0.00,0.00,0.00,0.00,211.35,8.11,151315,0,0,0,0,2010,3,5,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,0,64.88,4.00,"4,556.61",20.64,1.50,"1,601.01","3,288.25",192.01,8.68,118221,0,0,0,0,2012,9,28,False,True,False
421566,45,98,2012-10-05,628.10,0,64.89,3.98,"5,046.74",0.00,18.82,"2,253.43","2,340.01",192.17,8.67,118221,0,0,0,0,2012,10,5,False,True,False
421567,45,98,2012-10-12,"1,061.02",0,54.47,4.00,"1,956.28",0.00,7.89,599.32,"3,990.54",192.33,8.67,118221,0,0,0,0,2012,10,12,False,True,False
421568,45,98,2012-10-19,760.01,0,56.47,3.97,"2,004.02",0.00,3.18,437.73,"1,537.49",192.33,8.67,118221,0,0,0,0,2012,10,19,False,True,False


In [7]:
df = df.drop(['Unemployment', 'CPI', 'MarkDown5', 'MarkDown4'], axis=1)

In [8]:
df = df.drop(['IsSuperbowl', 'IsLaborDay', 'IsThanksgiving', 'Ischristmas'], axis=1)

In [9]:
train_data = df[:int(0.7*(len(df)))]
test_data = df[int(0.7*(len(df))):]

target = "Weekly_Sales"
used_cols = [c for c in df.columns.to_list() if c not in [target]] 

X_train = train_data[used_cols]
X_test = test_data[used_cols]
y_train = train_data[target]
y_test = test_data[target]

In [10]:
X_train = X_train.drop(['Date'], axis=1)
X_test = X_test.drop(['Date'], axis=1) 



In [11]:
def wmae_test(test, pred): # WMAE para test set
    weights = X_test['IsHoliday'].apply(lambda is_holiday:4 if is_holiday else 1)
    error = np.sum(weights * np.abs(test - pred), axis=0) / np.sum(weights)
    return error

In [21]:
gb = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=8,
                                min_samples_split = 10)

scaler=RobustScaler()

pipe = make_pipeline(scaler,gb)

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_train)

y_pred_test = pipe.predict(X_test)

print("WMAE on test set:", wmae_test(y_test, y_pred_test))

WMAE on test set: 5422.422045627405
